In [ ]:
# feature_extractor.py
import cv2, math
import mediapipe as mp
import numpy as np

mp_mesh = mp.solutions.face_mesh

def euclid(a, b): return math.hypot(a[0]-b[0], a[1]-b[1])

def extract_dta(img_path):
    img = cv2.imread(img_path)
    if img is None: return None
    h, w = img.shape[:2]
    with mp_mesh.FaceMesh(static_image_mode=True) as fm:
        res = fm.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        if not res.multi_face_landmarks: return None
        lm = res.multi_face_landmarks[0].landmark

    def pt(i): return (int(lm[i].x*w), int(lm[i].y*h))
    chin, forehead = pt(152), pt(10)
    jl, jr = pt(234), pt(454)
    cl, cr = pt(93), pt(323)

    features = [
        euclid(jl, jr),            # jaw width
        euclid(cl, cr),            # cheek width
        euclid(forehead, chin),    # face height
        math.degrees(math.atan2(jr[1]-jl[1], jr[0]-jl[0]))  # jaw angle
    ]
    return features


In [ ]:
# build_dataset.py
import os, csv
from feature_extractor import extract_dta

root = 'data/face-shape-dataset/'
with open('faces_dta.csv','w',newline='') as f:
    w = csv.writer(f)
    w.writerow(['jaw_w','cheek_w','face_h','angle','label'])
    for label in os.listdir(root):
        for img in os.listdir(os.path.join(root, label)):
            feats = extract_dta(os.path.join(root, label, img))
            if feats: w.writerow(feats + [label])


In [ ]:
# train_dta_model.py
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import joblib

df = pd.read_csv('faces_dta.csv')
X, y = df.iloc[:, :-1], df.iloc[:, -1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
print("Test accuracy:", clf.score(X_test, y_test))
joblib.dump(clf, 'dta_face_shape.pkl')


In [ ]:
# predict.py
import joblib
from feature_extractor import extract_dta

clf = joblib.load('dta_face_shape.pkl')
feats = extract_dta('your_photo.jpg')
print("Detected face shape:", clf.predict([feats])[0]) if feats else print("No face detected.")
